<a href="https://colab.research.google.com/github/swathi38538/BrickView-Project/blob/main/BrickView.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BrickView Real Estate Data Analysis

## STEP 1: Import Required Libraries

In [ ]:
import pandas as pd
import sqlite3
import json

## STEP 2: Load Dataset Files

In [ ]:
import pandas as pd

# Load all datasets
listings = pd.read_json("data/listings_final_expanded.json")
properties = pd.read_json("data/property_attributes_final_expanded.json")
agents = pd.read_json("data/agents_cleaned.json")
buyers = pd.read_json("data/buyers_cleaned.json")
sales = pd.read_csv("data/sales_cleaned.csv")

print("✅ All datasets loaded successfully!\n")

print("Listings Shape:", listings.shape)
print("Properties Shape:", properties.shape)
print("Agents Shape:", agents.shape)
print("Buyers Shape:", buyers.shape)
print("Sales Shape:", sales.shape)

✅ All datasets loaded successfully!

Listings Shape: (21200, 9)
Properties Shape: (21200, 13)
Agents Shape: (50, 9)
Buyers Shape: (20000, 7)
Sales Shape: (720, 4)


## STEP 3: Connect to SQLite Database

In [ ]:
import sqlite3

# Connect to SQLite Database
conn = sqlite3.connect("brickview.db")

print("✅ Connected to BrickView Database Successfully!")

✅ Connected to BrickView Database Successfully!


## STEP 4: Verify Tables in Database

In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

tables = cursor.fetchall()

print("Tables in the database:")

for table in tables:
    print(table[0])

Tables in the database:
Listings
Properties
Agents
Buyers
Sales


## Query 1: Average Listing Price by City

**Question:** What is the average listing price in each city?

In [ ]:
query1 = """
SELECT
    City,
    ROUND(AVG(Price),2) AS Average_Price
FROM Listings
GROUP BY City
ORDER BY Average_Price DESC;
"""

pd.read_sql_query(query1, conn)

,City,Average_Price
0,New York,2493319.74
1,Phoenix,2459961.76
2,Los Angeles,2442418.31
3,Houston,2436811.00
4,Chicago,2430677.90


## Query 2: Average Price per Square Foot by Property Type

**Question:** What is the average price per square foot for each property type?

In [ ]:
query2 = """
SELECT
    Property_Type,
    ROUND(AVG(Price / Sqft),2) AS Avg_Price_Per_Sqft
FROM Listings
GROUP BY Property_Type
ORDER BY Avg_Price_Per_Sqft DESC;
"""

pd.read_sql_query(query2, conn)

,Property_Type,Avg_Price_Per_Sqft
0,House,796.04
1,Apartment,792.20
2,Townhouse,789.75
3,Condo,754.68


## Query 3: Average Property Price by Furnishing Status

**Question:** How does furnishing status affect property prices?

In [ ]:
query3 = """
SELECT
    p.furnishing_status,
    ROUND(AVG(l.Price),2) AS Average_Price
FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id
GROUP BY p.furnishing_status
ORDER BY Average_Price DESC;
"""

pd.read_sql_query(query3, conn)

,furnishing_status,Average_Price
0,Furnished,2463539.06
1,Semi-Furnished,2461192.65
2,Unfurnished,2433975.10


## Query 4: Properties Near Metro Stations

**Question:** Do properties closer to metro stations command higher prices?

In [ ]:
query4 = """
SELECT
    CASE
        WHEN metro_distance_km <= 2 THEN 'Near Metro'
        ELSE 'Far from Metro'
    END AS Metro_Category,

    ROUND(AVG(l.Price), 2) AS Average_Price

FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id

GROUP BY Metro_Category;
"""

pd.read_sql_query(query4, conn)

,Metro_Category,Average_Price
0,Far from Metro,2464613.02
1,Near Metro,2377825.06


## Query 5: Rented vs Non-Rented Property Prices

**Question:** Are rented properties priced differently from non-rented ones?

In [ ]:
query5 = """
SELECT
    CASE
        WHEN p.is_rented = 1 THEN 'Rented'
        ELSE 'Not Rented'
    END AS Rental_Status,
    ROUND(AVG(l.Price), 2) AS Average_Price
FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id
GROUP BY p.is_rented;
"""

pd.read_sql_query(query5, conn)

,Rental_Status,Average_Price
0,Not Rented,2448250.3
1,Rented,2457184.9


## Query 6: Bedrooms and Bathrooms vs Property Pricing

**Question:** How do bedrooms and bathrooms affect pricing?

In [ ]:
query6 = """
SELECT
    p.bedrooms,
    p.bathrooms,
    ROUND(AVG(l.Price), 2) AS Average_Price
FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id
GROUP BY p.bedrooms, p.bathrooms
ORDER BY Average_Price DESC;
"""

pd.read_sql_query(query6, conn)

,bedrooms,bathrooms,Average_Price
0,2,4,2555766.23
1,5,4,2527191.97
2,1,2,2509474.77
3,2,3,2482618.93
4,3,4,2478881.22
5,5,2,2478068.17
6,4,4,2471535.90
7,1,3,2459032.59
8,2,1,2455735.36
9,1,4,2455532.96


## Query 7: Impact of Parking and Power Backup on Property Prices

**Question:** Do properties with parking and power backup sell at higher prices?

In [ ]:
query7 = """
SELECT
    parking_available,
    power_backup,
    ROUND(AVG(l.Price), 2) AS Average_Price
FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id
GROUP BY
    parking_available,
    power_backup
ORDER BY Average_Price DESC;
"""

pd.read_sql_query(query7, conn)

,parking_available,power_backup,Average_Price
0,1,1,2465651.62
1,1,0,2462363.80
2,0,1,2446292.59
3,0,0,2436677.75


## Query 8: Year Built vs Listing Price

**Question:** How does year built influence listing price?

In [ ]:
query8 = """
SELECT
    year_built,
    ROUND(AVG(l.Price), 2) AS Average_Price
FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id
GROUP BY year_built
ORDER BY year_built DESC;
"""

pd.read_sql_query(query8, conn)

,year_built,Average_Price
0,2023,2500120.42
1,2022,2397386.65
2,2021,2545272.16
3,2020,2465414.49
4,2019,2561310.32
5,2018,2385864.55
6,2017,2393633.46
7,2016,2428168.81
8,2015,2457272.07
9,2014,2358476.99


## Query 9: Cities with Highest Average Property Prices

**Question:** Which cities have the highest average property prices?

In [ ]:
query9 = """
SELECT
    City,
    ROUND(AVG(Price), 2) AS Average_Price
FROM Listings
GROUP BY City
ORDER BY Average_Price DESC;
"""

pd.read_sql_query(query9, conn)

,City,Average_Price
0,New York,2493319.74
1,Phoenix,2459961.76
2,Los Angeles,2442418.31
3,Houston,2436811.00
4,Chicago,2430677.90


## Query 10: Property Distribution by Price Buckets

**Question:** How are properties distributed across price buckets?

In [ ]:
query10 = """
SELECT
    CASE
        WHEN Price < 5000000 THEN 'Below 50 Lakhs'
        WHEN Price BETWEEN 5000000 AND 10000000 THEN '50 Lakhs - 1 Crore'
        ELSE 'Above 1 Crore'
    END AS Price_Bucket,
    COUNT(*) AS Total_Properties
FROM Listings
GROUP BY Price_Bucket
ORDER BY Total_Properties DESC;
"""

pd.read_sql_query(query10, conn)

,Price_Bucket,Total_Properties
0,Below 50 Lakhs,21200


## Query 11: Average Days on Market by City

**Question:** What is the average number of days properties remain on the market in each city?

In [ ]:
query11 = """
SELECT
    l.City,
    ROUND(AVG(s.Days_on_Market), 2) AS Average_Days_On_Market
FROM Listings l
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY l.City
ORDER BY Average_Days_On_Market DESC;
"""

pd.read_sql_query(query11, conn)

,City,Average_Days_On_Market
0,Los Angeles,65.13
1,Chicago,64.25
2,New York,60.85
3,Phoenix,59.65
4,Houston,58.51


## Query 12: Fastest Selling Property Types

**Question:** Which property types sell the fastest?

In [ ]:
query12 = """
SELECT
    l.Property_Type,
    ROUND(AVG(s.Days_on_Market), 2) AS Average_Days_On_Market
FROM Listings l
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY l.Property_Type
ORDER BY Average_Days_On_Market ASC;
"""

pd.read_sql_query(query12, conn)

,Property_Type,Average_Days_On_Market
0,House,58.34
1,Apartment,60.65
2,Townhouse,60.96
3,Condo,66.54


## Query 13: Percentage of Properties Sold Above Listing Price

**Question:** What percentage of properties were sold above the listing price?

In [ ]:
query13 = """
SELECT
    ROUND(
        (SUM(CASE WHEN s.Sale_Price > l.Price THEN 1 ELSE 0 END) * 100.0)
        / COUNT(*), 2
    ) AS Percentage_Sold_Above_Listing
FROM Listings l
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID;
"""

pd.read_sql_query(query13, conn)

,Percentage_Sold_Above_Listing
0,49.31


## Query 14: Sale-to-List Price Ratio by City

**Question:** What is the sale-to-list price ratio by city?

In [ ]:
query14 = """
SELECT
    l.City,
    ROUND(AVG((s.Sale_Price / l.Price) * 100), 2) AS Sale_To_List_Ratio
FROM Listings l
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY l.City
ORDER BY Sale_To_List_Ratio DESC;
"""

pd.read_sql_query(query14, conn)

,City,Sale_To_List_Ratio
0,Chicago,100.15
1,Houston,100.00
2,Los Angeles,99.98
3,New York,99.96
4,Phoenix,99.89


## Query 15: Listings That Took More Than 90 Days to Sell

**Question:** Which listings took more than 90 days to sell?

In [ ]:
query15 = """
SELECT
    l.Listing_ID,
    l.City,
    l.Property_Type,
    s.Days_on_Market
FROM Listings l
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
WHERE s.Days_on_Market > 90
ORDER BY s.Days_on_Market DESC;
"""

pd.read_sql_query(query15, conn)

,Listing_ID,City,Property_Type,Days_on_Market
0,L00157,Phoenix,Townhouse,120.018183
1,L00179,Chicago,Condo,120.011468
2,L00259,Phoenix,Apartment,120.010958
3,L00067,Chicago,Apartment,119.993220
4,L00287,Houston,Townhouse,119.970137
5,L00078,Los Angeles,House,119.008925
6,L01134,Chicago,Condo,119.006503
7,L00351,Phoenix,Condo,119.004860
8,L00057,Phoenix,Condo,119.000362
9,L00690,Phoenix,Townhouse,118.997563


## Query 16: Metro Distance vs Time on Market

**Question:** How does metro distance affect time on market?

In [ ]:
query16 = """
SELECT
    CASE
        WHEN p.metro_distance_km <= 2 THEN 'Near Metro'
        WHEN p.metro_distance_km <= 5 THEN 'Moderate Distance'
        ELSE 'Far from Metro'
    END AS Metro_Category,
    ROUND(AVG(s.Days_on_Market), 2) AS Average_Days_On_Market
FROM Listings l
INNER JOIN Properties p
ON l.Listing_ID = p.listing_id
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY Metro_Category
ORDER BY Average_Days_On_Market;
"""

pd.read_sql_query(query16, conn)

,Metro_Category,Average_Days_On_Market
0,Far from Metro,60.67
1,Moderate Distance,60.94
2,Near Metro,64.75


## Query 17: Monthly Sales Trend

**Question:** What is the monthly sales trend?

In [ ]:
query17 = """
SELECT
    strftime('%Y-%m', Date_Sold) AS Sale_Month,
    COUNT(*) AS Total_Sales
FROM Sales
GROUP BY Sale_Month
ORDER BY Sale_Month;
"""

pd.read_sql_query(query17, conn)

,Sale_Month,Total_Sales
0,2023-01,7
1,2023-02,17
2,2023-03,20
3,2023-04,36
4,2023-05,45
5,2023-06,31
6,2023-07,48
7,2023-08,50
8,2023-09,46
9,2023-10,64


## Query 18: Properties Currently Unsold

**Question:** Which properties are currently unsold?

In [ ]:
query18 = """
SELECT
    l.Listing_ID,
    l.City,
    l.Property_Type,
    l.Price
FROM Listings l
LEFT JOIN Sales s
ON l.Listing_ID = s.Listing_ID
WHERE s.Listing_ID IS NULL
LIMIT 20;
"""

pd.read_sql_query(query18, conn)

,Listing_ID,City,Property_Type,Price
0,L00002,Los Angeles,Apartment,1.519141e+06
1,L00005,Phoenix,Townhouse,5.622970e+05
2,L00009,New York,Townhouse,1.224551e+06
3,L00014,Chicago,Apartment,1.377440e+06
4,L00015,Los Angeles,House,1.069429e+06
5,L00016,Phoenix,House,1.535740e+06
6,L00017,New York,House,1.823444e+06
7,L00018,Houston,Condo,2.388070e+05
8,L00020,Houston,House,6.554930e+05
9,L00021,Phoenix,Condo,1.666601e+06


## Query 19: Which agents have closed the most sales?

**Question:** Which agents have closed the most sales?

In [ ]:
query19 = """
SELECT
    a.Name,
    COUNT(s.Listing_ID) AS Total_Sales
FROM Agents a
INNER JOIN Listings l
ON a.Agent_ID = l.Agent_ID
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY a.Agent_ID, a.Name
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query19, conn)

,Name,Total_Sales
0,Agent A0042,25
1,Agent A0011,24
2,Agent A0014,21
3,Agent A0035,21
4,Agent A0043,20
5,Agent A0046,20
6,Agent A0007,19
7,Agent A0048,19
8,Agent A0027,18
9,Agent A0029,18


## Query 20: Who are the top agents by total sales revenue?

**Question:** Who are the top agents by total sales revenue?

In [ ]:
query20 = """
SELECT
    a.Name,
    ROUND(SUM(s.Sale_Price),2) AS Total_Sales_Revenue
FROM Agents a
INNER JOIN Listings l
ON a.Agent_ID = l.Agent_ID
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY a.Agent_ID, a.Name
ORDER BY Total_Sales_Revenue DESC;
"""

pd.read_sql_query(query20, conn)

,Name,Total_Sales_Revenue
0,Agent A0011,27882272.02
1,Agent A0042,27191605.99
2,Agent A0043,24102418.02
3,Agent A0035,22725751.99
4,Agent A0014,22034008.02
5,Agent A0046,21335804.97
6,Agent A0048,21186306.00
7,Agent A0027,21099696.96
8,Agent A0009,20279274.97
9,Agent A0029,19586498.02


## Query 21: Which agents close deals fastest?

**Question:** Which agents close deals fastest?

In [ ]:
query21 = """
SELECT
    Name,
    avg_closing_days
FROM Agents
ORDER BY avg_closing_days ASC;
"""

pd.read_sql_query(query21, conn)

,Name,avg_closing_days
0,Agent A0048,15
1,Agent A0035,20
2,Agent A0042,21
3,Agent A0023,23
4,Agent A0045,26
5,Agent A0041,27
6,Agent A0016,31
7,Agent A0034,31
8,Agent A0028,33
9,Agent A0032,33


## Query 22: Does experience correlate with deals closed?

**Question:** Does experience correlate with deals closed?

In [ ]:
query22 = """
SELECT
    experience_years,
    AVG(deals_closed) AS Average_Deals_Closed
FROM Agents
GROUP BY experience_years
ORDER BY experience_years;
"""

pd.read_sql_query(query22, conn)

,experience_years,Average_Deals_Closed
0,1,262.000000
1,2,50.000000
2,3,14.000000
3,4,140.000000
4,5,88.333333
5,7,73.000000
6,9,283.000000
7,10,93.500000
8,11,109.000000
9,12,257.000000


## Query 23: Do agents with higher ratings close deals faster?

**Question:** Do agents with higher ratings close deals faster?

In [ ]:
query23 = """
SELECT
    rating,
    ROUND(AVG(avg_closing_days),2) AS Average_Closing_Days
FROM Agents
GROUP BY rating
ORDER BY rating DESC;
"""

pd.read_sql_query(query23, conn)

,rating,Average_Closing_Days
0,5.0,53.50
1,4.9,54.50
2,4.8,51.00
3,4.6,36.00
4,4.5,61.00
5,4.4,38.00
6,4.3,58.80
7,4.2,26.00
8,4.1,55.83
9,4.0,27.00


## Query 24: What is the average commission earned by each agent?

**Question:** What is the average commission earned by each agent?

In [ ]:
query24 = """
SELECT
    a.Name,
    ROUND(AVG(s.Sale_Price * a.commission_rate / 100), 2) AS Average_Commission
FROM Agents a
INNER JOIN Listings l
ON a.Agent_ID = l.Agent_ID
INNER JOIN Sales s
ON l.Listing_ID = s.Listing_ID
GROUP BY a.Agent_ID, a.Name
ORDER BY Average_Commission DESC;
"""

pd.read_sql_query(query24, conn)

,Name,Average_Commission
0,Agent A0009,33997.61
1,Agent A0002,32326.16
2,Agent A0035,32032.49
3,Agent A0012,31737.61
4,Agent A0030,31665.09
5,Agent A0018,31438.36
6,Agent A0031,30358.05
7,Agent A0050,28564.40
8,Agent A0017,28406.74
9,Agent A0024,27805.90


## Query 25: Which agents currently have the most active listings?

**Question:** Which agents currently have the most active listings?

In [ ]:
query25 = """
SELECT
    a.Name,
    COUNT(l.Listing_ID) AS Active_Listings
FROM Agents a
INNER JOIN Listings l
ON a.Agent_ID = l.Agent_ID
LEFT JOIN Sales s
ON l.Listing_ID = s.Listing_ID
WHERE s.Listing_ID IS NULL
GROUP BY a.Agent_ID, a.Name
ORDER BY Active_Listings DESC;
"""

pd.read_sql_query(query25, conn)

,Name,Active_Listings
0,Agent A0023,446
1,Agent A0011,439
2,Agent A0008,438
3,Agent A0042,435
4,Agent A0014,432
5,Agent A0044,430
6,Agent A0020,426
7,Agent A0012,425
8,Agent A0015,425
9,Agent A0048,425


## Query 26: What percentage of buyers are investors vs end users?

**Question:** What percentage of buyers are investors vs end users?

In [ ]:
query26 = """
SELECT
    buyer_type,
    COUNT(*) AS Total_Buyers,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM Buyers), 2) AS Percentage
FROM Buyers
GROUP BY buyer_type
ORDER BY Percentage DESC;
"""

pd.read_sql_query(query26, conn)

,buyer_type,Total_Buyers,Percentage
0,End User,10020,50.1
1,Investor,9980,49.9


## Query 27: Which cities have the highest loan uptake rate?

**Question:** Which cities have the highest loan uptake rate?

In [ ]:
query27 = """
SELECT
    l.City,
    ROUND(AVG(b.loan_taken) * 100, 2) AS Loan_Uptake_Rate
FROM Buyers b
INNER JOIN Listings l
ON b.sale_id = l.Listing_ID
GROUP BY l.City
ORDER BY Loan_Uptake_Rate DESC;
"""

pd.read_sql_query(query27, conn)

,City,Loan_Uptake_Rate
0,Los Angeles,50.99
1,Chicago,50.57
2,New York,50.22
3,Houston,49.71
4,Phoenix,49.50


## Query 28: What is the average loan amount by buyer type?

**Question:** What is the average loan amount by buyer type?

In [ ]:
query28 = """
SELECT
    buyer_type,
    ROUND(AVG(loan_amount), 2) AS Average_Loan_Amount
FROM Buyers
WHERE loan_taken = 1
GROUP BY buyer_type
ORDER BY Average_Loan_Amount DESC;
"""

pd.read_sql_query(query28, conn)

,buyer_type,Average_Loan_Amount
0,Investor,5211545.97
1,End User,5210307.45


## Query 29: Which payment mode is most commonly used?

**Question:** Which payment mode is most commonly used?

In [ ]:
query29 = """
SELECT
    payment_mode,
    COUNT(*) AS Total_Transactions
FROM Buyers
GROUP BY payment_mode
ORDER BY Total_Transactions DESC;
"""

pd.read_sql_query(query29, conn)

,payment_mode,Total_Transactions
0,Cash,5088
1,UPI,5012
2,Cheque,4951
3,Bank Transfer,4949


## Query 30: Do loan-backed purchases take longer to close?

**Question:** Do loan-backed purchases take longer to close?

In [ ]:
query30 = """
SELECT
    CASE
        WHEN b.loan_taken = 1 THEN 'Loan Taken'
        ELSE 'No Loan'
    END AS Loan_Status,
    ROUND(AVG(s.Days_on_Market), 2) AS Average_Days_On_Market
FROM Buyers b
INNER JOIN Sales s
ON b.sale_id = s.Listing_ID
GROUP BY Loan_Status;
"""

pd.read_sql_query(query30, conn)

,Loan_Status,Average_Days_On_Market
0,Loan Taken,62.15
1,No Loan,61.07


In [ ]:

import pandas as pd
import sqlite3

conn = sqlite3.connect("brickview.db")

print(pd.read_sql_query("SELECT * FROM Listings LIMIT 5", conn))

conn.close()


  Listing_ID         City Property_Type         Price         Sqft  \
0     L00001     New York     Apartment  1.655144e+06  2753.009121   
1     L00002  Los Angeles     Apartment  1.519141e+06  4966.988193   
2     L00003      Houston     Apartment  1.624890e+05  1267.003959   
3     L00004      Phoenix     Apartment  1.277016e+06  2128.014429   
4     L00005      Phoenix     Townhouse  5.622970e+05  4178.997421   

  Date_Listed Agent_ID   Latitude   Longitude  
0  2023-05-06    A0015  33.965208  -69.861589  
1  2023-02-14    A0038  42.547892  -90.277860  
2  2023-04-22    A0015  28.732327 -115.952982  
3  2024-01-02    A0042  26.403938  -74.771490  
4  2023-10-29    A0018  39.425252  -83.917878  
